### Packages

In [ ]:
import warnings

import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

warnings.filterwarnings("ignore")

### Loading Dataset

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

X = train_df.drop(columns=["sold_first_hour"])
y = train_df["sold_first_hour"]

# split up training set into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_test = test_df.copy()

print("train_df:", train_df.shape)
print("test_df:", test_df.shape)
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


print(X_train.head(5))

### Data exploration

In [ ]:
print("Feature scales:\n")

for col in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[col]):
        print(
            f"{col}: min={X_train[col].min()}, max={X_train[col].max()}, "
            f"mean={X_train[col].mean():.2f}, std={X_train[col].std():.2f}"
        )
    else:
        print(f"{col}: categories={sorted(X_train[col].unique())}")

### Feature extraction

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=["object", "string"]).columns.tolist()

# seatid is an identifier, so leave it out of the model features
if "seatid" in numeric_cols:
    numeric_cols.remove("seatid")

feature_cols = numeric_cols + categorical_cols

X_train_features = pd.get_dummies(X_train[feature_cols], columns=categorical_cols, dtype=float)
X_val_features = pd.get_dummies(X_val[feature_cols], columns=categorical_cols, dtype=float)
X_test_features = pd.get_dummies(X_test[feature_cols], columns=categorical_cols, dtype=float)

X_val_features = X_val_features.reindex(columns=X_train_features.columns, fill_value=0)
X_test_features = X_test_features.reindex(columns=X_train_features.columns, fill_value=0)

X_train_matrix = X_train_features.to_numpy()
X_val_matrix = X_val_features.to_numpy()
X_test_matrix = X_test_features.to_numpy()

y_train_array = y_train.to_numpy()
y_val_array = y_val.to_numpy()

print("Feature matrix shapes:")
print("X_train_matrix:", X_train_matrix.shape)
print("X_val_matrix:", X_val_matrix.shape)
print("X_test_matrix:", X_test_matrix.shape)
print("Number of features:", len(X_train_features.columns))

X_train_features.head()

### Scaling features

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_matrix)
X_val_scaled = scaler.transform(X_val_matrix)
X_test_scaled = scaler.transform(X_test_matrix)


pd.DataFrame(X_train_scaled, columns=X_train_features.columns).head()

### Creating model

In [ ]:
def make_svm(kernel="rbf", C=1.0, degree=3, gamma="scale", coef0=0.0):
    return SVC(
        kernel=kernel,
        C=C,
        degree=degree,
        gamma=gamma,
        coef0=coef0,
    )

### Grid search

In [ ]:
grid_search_sample_size = 20000

X_search, _, y_search, _ = train_test_split(
    X_train_scaled,
    y_train_array,
    train_size=grid_search_sample_size,
    stratify=y_train_array,
    random_state=42,
)

param_grid = [
    {
        "kernel": ["linear"],
        "C": [0.01, 0.1, 1, 10],
    },
    {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.01, 0.1, 1],
    },
    {
        "kernel": ["poly"],
        "C": [0.1, 1],
        "degree": [2, 3, 4],
        "gamma": ["scale", 0.01, 0.1],
        "coef0": [0.0, 0.5],
    },
    {
        "kernel": ["sigmoid"],
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.01, 0.1],
        "coef0": [0.0, 0.5, 1.0],
    },
]

grid_search = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_search, y_search)

best_params = grid_search.best_params_
best_svm = SVC(**best_params)
best_svm.fit(X_train_scaled, y_train_array)
val_accuracy = best_svm.score(X_val_scaled, y_val_array)

print("Grid-search sample size:", X_search.shape[0])
print("Best parameters:", best_params)
print("Best cross-validation accuracy:", f"{grid_search.best_score_:.4f}")
print("Validation accuracy:", f"{val_accuracy:.4f}")

pd.DataFrame(grid_search.cv_results_)[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score").head(10)